# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefelshazly/FlyRankMachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: If CTR is below 50% of the median for that position tier, flag for title/meta rewrite. If engagement is below median but clicks are above median, flag for content update.

Reason codes: ctr_underperforming, low_engagement_high_clicks, none

In [24]:
!git clone https://github.com/yousefelshazly/FlyRankMachineLearning.git
%cd FlyRankMachineLearning

Cloning into 'FlyRankMachineLearning'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 129 (delta 41), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.85 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/FlyRankMachineLearning/FlyRankMachineLearning


In [25]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Shape: {df.shape}")
print(df.columns.tolist())
print(df[['ctr', 'avg_position', 'impressions_last_30d', 'search_volume']].head(10))

Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
    ctr  avg_position  impressions_last_30d  search_volume
0  0.76          10.6                   578           10.0
1  0.05          20.3                  2501           90.0
2  0.09          36.5            

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [29]:
import numpy as np
import pandas as pd
import os

# ===== STEP 1: Load and filter data =====
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df_clean = df[df['avg_position'] > 0].copy()
print(f"Rows after filtering: {len(df_clean)}")

# ===== STEP 2: Create position tiers =====
df_clean['position_tier'] = pd.cut(
    df_clean['avg_position'],
    bins=[0, 5, 10, 20, 50, 100],
    labels=['1-5', '6-10', '11-20', '21-50', '51-100']
)

# ===== STEP 3: Compute median CTR per tier =====
df_clean['median_ctr_for_tier'] = df_clean.groupby('position_tier', observed=True)['ctr'].transform('median')
print("Median CTR per tier:")
print(df_clean.groupby('position_tier', observed=True)['median_ctr_for_tier'].first())

# ===== STEP 4: Compute engagement medians =====
df_eng = df_clean[df_clean['engagement_rate'] > 0].copy()
engagement_median = df_eng['engagement_rate'].median()
clicks_median = df_eng['clicks_last_30d'].median()
print(f"Engagement median: {engagement_median}, Clicks median: {clicks_median}")

# ===== STEP 5: Create boolean flags =====
df_clean['is_ctr_underperforming'] = df_clean['ctr'] < (0.5 * df_clean['median_ctr_for_tier'])

df_clean['is_engagement_issue'] = (
    (df_clean['engagement_rate'] > 0) &
    (df_clean['engagement_rate'] < engagement_median) &
    (df_clean['clicks_last_30d'] >= clicks_median)
)

# ===== STEP 6: Compute normalized scores =====
max_impressions = df_clean['impressions_last_30d'].max()
df_clean['impressions_score'] = (df_clean['impressions_last_30d'] / max_impressions) * 100

max_clicks = df_clean['clicks_last_30d'].max()
df_clean['clicks_score'] = (df_clean['clicks_last_30d'] / max_clicks) * 100

# ===== STEP 7: Combine into final score =====
df_clean['score'] = np.select(
    condlist=[
        df_clean['is_ctr_underperforming'],
        df_clean['is_engagement_issue']
    ],
    choicelist=[
        df_clean['impressions_score'],
        df_clean['clicks_score']
    ],
    default=0
)

# ===== STEP 8: Assign reason codes =====
df_clean['reason_code'] = np.select(
    condlist=[
        df_clean['is_ctr_underperforming'],
        df_clean['is_engagement_issue']
    ],
    choicelist=[
        'ctr_underperforming',
        'low_engagement_high_clicks'
    ],
    default='none'
)

# ===== STEP 9: Assign action labels =====
df_clean['action'] = np.select(
    condlist=[
        df_clean['is_ctr_underperforming'],
        df_clean['is_engagement_issue']
    ],
    choicelist=[
        'rewrite_title_meta',
        'content_update'
    ],
    default='monitor'
)

# ===== STEP 10: Sort by score (ranked queue) =====
ranked_queue = df_clean.sort_values('score', ascending=False)[[
    'content_id', 'client_id', 'score', 'reason_code', 'action',
    'ctr', 'avg_position', 'impressions_last_30d', 'engagement_rate', 'clicks_last_30d'
]].reset_index(drop=True)

print(f"\nRanked queue shape: {ranked_queue.shape}")
print("\nTop 20 rows:")
print(ranked_queue.head(20))

# ===== STEP 11: Save CSV =====
from google.colab import drive
drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/seo-outputs', exist_ok=True)
ranked_queue.to_csv('/content/drive/MyDrive/seo-outputs/baseline_action_score.csv', index=False)
print("Saved to Google Drive")

Rows after filtering: 28795
Median CTR per tier:
position_tier
1-5       0.18
6-10      0.14
11-20     0.10
21-50     0.03
51-100    0.00
Name: median_ctr_for_tier, dtype: float64
Engagement median: 4.76, Clicks median: 4.0

Ranked queue shape: (28795, 10)

Top 20 rows:
              content_id          client_id      score  \
0   content_8451fc6f034d  client_d029fa3a95  70.754116   
1   content_89e84d699e9e  client_349c41201b  60.459184   
2   content_8e7ba84a972b  client_7f2253d7e2  54.676871   
3   content_44e481c8f55b  client_19581e27de  52.976190   
4   content_4a6607efcb46  client_6208ef0f77  51.216520   
5   content_2c2606c5d176  client_19581e27de  49.149660   
6   content_34e549c30fa0  client_6208ef0f77  47.789116   
7   content_36ff89c8214e  client_19581e27de  44.801839   
8   content_db5989a78dd3  client_4e07408562  42.602041   
9   content_c9139af55035  client_19581e27de  42.517007   
10  content_c84a0ab98e90  client_f369cb89fc  41.599106   
11  content_83e2682b8f27  client_

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:

Rows 0-5, 11, 17 (CTR underperforming): Action = rewrite_title_meta, Reason = ctr_underperforming. Confidence: High if impressions are high. Could be wrong if page is new or competition is too strong.

Rows 6-10, 12-16, 18-19 (Low engagement + high clicks): Action = content_update, Reason = low_engagement_high_clicks. Confidence: Medium. Could be wrong if users find answers quickly without scrolling.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Three weak picks (from top 20):

content_91652435f57a: Engagement is 0.0 — no GA4 data. Action is based only on CTR, which could be wrong diagnosis. Fix: Require minimum engagement data before scoring.

content_34e549c30fa0: Engagement is 3.1 (65% of median) — not actually "low." Flag is too sensitive. Fix: Use stricter threshold (e.g., < 50% of median).

content_83e2682b8f27: Engagement is 3.84 (81% of median) — barely qualifies as "low." Fix: Require engagement < 2.5 (not just < 4.76).

Leakage check: No leakage — all signals (CTR, engagement, clicks, impressions) are from the same 30-day window and would exist at decision time. No future data or internal product flags used.



In [30]:
# Check top 20 for rows with engagement = 0 or very low impressions
ranked_queue.head(20)[['content_id', 'score', 'reason_code', 'engagement_rate', 'impressions_last_30d', 'clicks_last_30d']]

,content_id,score,reason_code,engagement_rate,impressions_last_30d,clicks_last_30d
0,content_8451fc6f034d,70.754116,ctr_underperforming,2.02,168958,36
1,content_89e84d699e9e,60.459184,low_engagement_high_clicks,0.70,83350,711
2,content_8e7ba84a972b,54.676871,low_engagement_high_clicks,1.19,75144,643
3,content_44e481c8f55b,52.976190,low_engagement_high_clicks,0.76,104458,623
4,content_4a6607efcb46,51.216520,ctr_underperforming,2.30,122303,10
5,content_2c2606c5d176,49.149660,low_engagement_high_clicks,1.30,104248,578
6,content_34e549c30fa0,47.789116,low_engagement_high_clicks,3.10,37321,562
7,content_36ff89c8214e,44.801839,ctr_underperforming,1.68,106985,35
8,content_db5989a78dd3,42.602041,low_engagement_high_clicks,2.32,238796,501
9,content_c9139af55035,42.517007,low_engagement_high_clicks,1.62,47091,500


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.